In [0]:
%sql

create schema if not exists healthcare_endtoend_dev.silver
Managed location 'abfss://unity-catalog-root@stazdatabricksendtoend.dfs.core.windows.net/healthcare-endtoend-dev/app-data/silver'

In [0]:
%sql

select * from healthcare_endtoend_dev.bronze.visit_raw;

In [0]:
bronze_table = 'healthcare_endtoend_dev.bronze.visit_raw'
silver_table = 'healthcare_endtoend_dev.silver.fact_visit'
checkpoint_path = "abfss://unity-catalog-root@stazdatabricksendtoend.dfs.core.windows.net/healthcare-endtoend-dev/silver/fact_visit/checkpoint/"

silver_patient_table = "healthcare_endtoend_dev.silver.dim_patient"
silver_hospital_table = "healthcare_endtoend_dev.silver.dim_hospital"
silver_diagnosis_table = "healthcare_endtoend_dev.silver.dim_diagnosis"
bronze_table = "healthcare_endtoend_dev.bronze.visit_raw"

In [0]:

df_patient = spark.read.table(silver_patient_table)
df_hospital = spark.read.table(silver_hospital_table)
df_diagnosis = spark.read.table(silver_diagnosis_table) 

In [0]:
df_visit_bronze = (
    spark.readStream.table(bronze_table)
)

In [0]:
df_patient.display()
df_hospital.display()
df_diagnosis.display()

In [0]:
# Rename columns to avoid duplicates
df_patient = df_patient.withColumnRenamed("city", "patient_city") \
            .withColumnRenamed("load_timestamp", "patient_load_timestamp") 
df_diagnosis = df_diagnosis.withColumnRenamed("load_timestamp", "diagnosis_load_timestamp")

df_hospital = df_hospital.withColumnRenamed("city", "hospital_city") \
                        .withColumnRenamed("load_timestamp","hospital_load_timestamp")

df_patient.show(10)
df_diagnosis.show(10)
df_hospital.show(10)

In [0]:

from pyspark.sql.functions import col, lag, to_date, datediff, current_timestamp, expr
from delta.tables import DeltaTable

# Rename columns to avoid duplicates
df_patient = df_patient.withColumnRenamed("city", "patient_city") \
            .withColumnRenamed("load_timestamp", "patient_load_timestamp") 
df_diagnosis = df_diagnosis.withColumnRenamed("load_timestamp", "diagnosis_load_timestamp")

df_hospital = df_hospital.withColumnRenamed("city", "hospital_city") \
                        .withColumnRenamed("load_timestamp","hospital_load_timestamp")


# Join clean fact visit with dimension tables
df_fact_combined = (
    df_visit_bronze
        .filter(col("admission_date") != "admission_date")
        .join(df_patient, "patient_id", "left")
        .join(df_hospital, "hospital_id", "left")
        .join(df_diagnosis, "diagnosis_code", "left")
        .withColumn("admission_date", expr("try_cast(admission_date as DATE)"))
        .withColumn("discharge_date", expr("try_cast(discharge_date as DATE)"))
        .withColumn("load_timestamp", current_timestamp())
)
# -------------------------
# Merge into Silver fact_visit
# -------------------------
def merge_fact_visit(batch_df, batch_id):
    # CRITICAL: Get the spark session from the dataframe itself
    batch_spark = batch_df.sparkSession

    print(f"Processing Batch: {batch_id}")
    print(f"Columns in batch: {batch_df.columns}")

    if not batch_spark.catalog.tableExists(silver_table):
        batch_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
        return

    fact = DeltaTable.forName(batch_spark, silver_table)
    fact.alias("t").merge(
        batch_df.alias("s"),
        "t.visit_id = s.visit_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()


# -------------------------
# Run as availableNow incremental
# -------------------------
try:
    query = (
        df_fact_combined.writeStream
            .foreachBatch(merge_fact_visit)
            .outputMode("update")
            .trigger(availableNow=True)
            .option("checkpointLocation", checkpoint_path)
            .start()
    )
    query.awaitTermination()
except Exception as e:
    print(f"Error in merge: {str(e)}")
    raise e

In [0]:
%sql

select * from healthcare_endtoend_dev.silver.fact_visit;